# AMPS II · Week 04 — HW4
## J2 plasticity: return mapping and tangent, hardening, autofrettage, and an LLM-generated plasticity script

**Due before Week 5 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q4 runs an LLM-generated script that is **meant** to be wrong.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Radial return, the consistent tangent, first yield and limit load of the tube (theory) | 20 |
| 2 | Hardening: the material point against the bilinear curve, the tube beyond $p_{\rm lim}$, Newton rates | 25 |
| 3 | Autofrettage: overload, unload, residual stress, reverse yielding in a thick tube ($b/a=3$) | 35 |
| 4 | Debugging an LLM-generated plasticity script | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model modules from the lab (unchanged). Run both cells once.

In [ ]:
%%writefile /tmp/fnx/j2.py
"""J2 (von Mises) plasticity with linear isotropic hardening at the material points -- NumPy only.
Mandel vectors (xx, yy, zz, sqrt(2) xy) for stress and strain, so that sigma . eps = sigma : eps and C is a 4 x 4 matrix.
update() is the backward-Euler return mapping for MANY points at once (arrays of shape (n, 4))."""
import numpy as np

ONE = np.array([1.0, 1.0, 1.0, 0.0])                 # the identity tensor in Mandel form
IDEV = np.eye(4) - np.outer(ONE, ONE) / 3.0          # deviatoric projector: s = IDEV @ sigma

def dev(sig):
    return sig @ IDEV                                # IDEV is symmetric

def von_mises(sig):
    s = dev(sig)
    return np.sqrt(1.5 * np.einsum("ij,ij->i", s, s))

class J2:
    """E, nu: elasticity; sig0: initial yield stress; H: linear isotropic hardening modulus, sigma_y = sig0 + H p"""
    def __init__(self, E, nu, sig0, H=0.0):
        self.E, self.nu, self.sig0, self.H = E, nu, sig0, H
        self.mu = E / (2 * (1 + nu)); self.lmbda = E * nu / ((1 + nu) * (1 - 2 * nu))
        self.C = self.lmbda * np.outer(ONE, ONE) + 2 * self.mu * np.eye(4)

    def update(self, deps, sig_n, p_n, tangent="consistent"):
        """deps: strain increment since the last CONVERGED step, sig_n, p_n: stress and p at that step.
        Returns sig, p, Ct (n, 4, 4) and the mask of plastic points. tangent = "consistent" or "elastic"."""
        mu, H = self.mu, self.H
        sig_tr = sig_n + deps @ self.C                   # elastic trial stress
        s = dev(sig_tr)
        seq = np.sqrt(1.5 * np.einsum("ij,ij->i", s, s))
        f = seq - (self.sig0 + H * p_n)                  # trial yield function
        pl = f > 0.0
        dp = np.where(pl, f / (3 * mu + H), 0.0)          # plastic multiplier = increment of p
        safe = np.where(seq > 0.0, seq, 1.0)
        nhat = s / (np.sqrt(2.0 / 3.0) * safe)[:, None]  # unit normal s/|s|
        sig = sig_tr - (2 * mu * np.sqrt(1.5) * dp)[:, None] * nhat   # radial return
        Ct = np.broadcast_to(self.C, (len(f), 4, 4)).copy()
        if tangent == "consistent":
            b = 3 * mu * dp / safe                       # 1 - theta
            tb = np.where(pl, 3 * mu / (3 * mu + H) - b, 0.0)
            Ct -= 2 * mu * (b[:, None, None] * IDEV + tb[:, None, None] * np.einsum("ni,nj->nij", nhat, nhat))
        return sig, p_n + dp, Ct, pl

def uniaxial(mat, exx, tangent="consistent", tol=1e-10, maxit=50):
    """uniaxial STRESS test of one material point: prescribed history of eps_xx, sigma_yy = sigma_zz = sigma_xy = 0 by a local
    Newton iteration on the three other strain components (with the tangent of update()). Returns columns sigma_xx, p, iterations."""
    sig, p, eps, out = np.zeros((1, 4)), np.zeros(1), np.zeros(4), []
    for e in exx:
        deps = np.zeros(4); deps[0] = e - eps[0]; deps[1] = deps[2] = -mat.nu * deps[0]     # elastic guess
        for it in range(maxit):
            s, pn, Ct, pl = mat.update(deps[None], sig, p, tangent)
            r = s[0, 1:]
            if np.abs(r).max() <= tol * mat.sig0:
                break
            deps[1:] -= np.linalg.solve(Ct[0][1:, 1:], r)
        eps, sig, p = eps + deps, s, pn
        out.append((s[0, 0], pn[0], it))
    return np.array(out)

In [ ]:
%%writefile /tmp/fnx/plas.py
"""Small-strain J2 plasticity in DOLFINx 0.11: quadrature spaces for stress, p and the tangent, an explicit Newton loop.
Two geometries of the same thick tube in plane strain: the 2D quarter cross-section (curved mapped mesh, no Gmsh) and
the 1D radial model (an interval in r, weight r) used as the reference. Load: inner pressure or inner radial displacement."""
import os, sys
import numpy as np
import ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, create_matrix, create_vector, set_bc
sys.path.insert(0, "/tmp/fnx")
from j2 import J2, ONE, IDEV, dev, von_mises

comm = MPI.COMM_WORLD
os.makedirs("/tmp/fnx", exist_ok=True)
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

INNER, OUTER, BOTTOM, LEFT = 1, 2, 3, 4

def annulus_mesh(ri, ro, nr, nt):
    """quarter annulus, nr x nt cells in (r, theta) split in triangles, 6-node curved cells mapped from (r, theta) (as Week 3).
    Facet tags: 1 r = ri, 2 r = ro, 3 y = 0, 4 x = 0."""
    if comm.rank == 0:
        nI, nJ = 2 * nr + 1, 2 * nt + 1
        R, TH = np.meshgrid(np.linspace(ri, ro, nI), np.linspace(0, np.pi / 2, nJ), indexing="ij")
        x = np.c_[(R * np.cos(TH)).ravel(), (R * np.sin(TH)).ravel()]
        x[np.abs(x) < 1e-14] = 0.0
        node = lambda p: p[0] * nJ + p[1]
        mid = lambda p, q: ((p[0] + q[0]) // 2, (p[1] + q[1]) // 2)
        cells = []
        for a in range(nr):
            for b in range(nt):
                i, j = 2 * a, 2 * b
                for p, q, s in (((i, j), (i + 2, j), (i + 2, j + 2)), ((i, j), (i + 2, j + 2), (i, j + 2))):
                    cells.append([node(p), node(q), node(s), node(mid(q, s)), node(mid(p, s)), node(mid(p, q))])
        cells = np.array(cells, dtype=np.int64)
    else:
        x, cells = np.zeros((0, 2)), np.zeros((0, 6), dtype=np.int64)
    msh = mesh.create_mesh(comm, cells, ufl.Mesh(basix.ufl.element("Lagrange", "triangle", 2, shape=(2,))), x)
    rad = lambda X: np.hypot(X[0], X[1])
    return msh, _tags(msh, [(INNER, lambda X: np.isclose(rad(X), ri)), (OUTER, lambda X: np.isclose(rad(X), ro)),
                            (BOTTOM, lambda X: np.isclose(X[1], 0.0)), (LEFT, lambda X: np.isclose(X[0], 0.0))])

def radial_mesh(ri, ro, n):
    """interval ri <= r <= ro (the 1D radial model). Facet (= point) tags: 1 r = ri, 2 r = ro."""
    msh = mesh.create_interval(comm, n, [ri, ro])
    return msh, _tags(msh, [(INNER, lambda X: np.isclose(X[0], ri)), (OUTER, lambda X: np.isclose(X[0], ro))])

def _tags(msh, marks):
    fdim = msh.topology.dim - 1
    fac, val = [], []
    for tag, f in marks:
        e = mesh.locate_entities_boundary(msh, fdim, f); fac.append(e); val.append(np.full(len(e), tag, dtype=np.int32))
    fac, idx = np.unique(np.concatenate(fac), return_index=True)
    return mesh.meshtags(msh, fdim, fac, np.concatenate(val)[idx])

# ── exact results for the tube (plane strain) ──────────────────────────────────────────────────────────────────
def lame_ur(p, ri, ro, E, nu, r=None):
    """elastic plane-strain radial displacement u_r(r) of a tube under inner pressure p (r = ri by default)"""
    r = ri if r is None else r
    A = p * ri**2 / (ro**2 - ri**2); B = A * ro**2
    return (1 + nu) / E * ((1 - 2 * nu) * A * r + B / r)

def p_first_yield(sig0, ri, ro, nu):
    """first yield at r = ri, von Mises, plane strain (sigma_zz = 2 nu A). nu = 0.5 gives sig0/sqrt(3) (1 - ri^2/ro^2)."""
    return sig0 * (ro**2 - ri**2) / np.sqrt(3 * ro**4 + (1 - 2 * nu)**2 * ri**4)

def p_limit(sig0, ri, ro):
    return 2 * sig0 / np.sqrt(3) * np.log(ro / ri)

# ── the model ──────────────────────────────────────────────────────────────────────────────────────────────────
class Tube:
    """msh, ft: annulus_mesh (2D) or radial_mesh (1D); mat: J2. qdeg: quadrature degree of the internal variables.
    control = "pressure" (inner pressure p) or "displacement" (inner radial displacement)."""
    def __init__(self, msh, ft, mat, ri, ro, deg=2, qdeg=2, control="pressure", tangent="consistent"):
        self.msh, self.mat, self.ri, self.ro, self.tangent, self.control = msh, mat, ri, ro, tangent, control
        self.dim = msh.topology.dim
        cell = msh.basix_cell()
        if self.dim == 2:
            self.V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
        else:
            self.V = fem.functionspace(msh, ("Lagrange", deg))
        Q4 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4,), degree=qdeg))
        Q44 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4, 4), degree=qdeg))
        Q1 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(), degree=qdeg))
        self.sig, self.sig_n, self.Ct = fem.Function(Q4), fem.Function(Q4), fem.Function(Q44)
        self.p, self.p_n = fem.Function(Q1), fem.Function(Q1)
        self.Ct.x.array[:] = np.tile(mat.C.ravel(), len(self.Ct.x.array) // 16)
        self.u, self.Du, self.du = fem.Function(self.V), fem.Function(self.V), fem.Function(self.V)
        self.dx = ufl.Measure("dx", domain=msh, metadata={"quadrature_degree": qdeg})
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=ft)
        x = ufl.SpatialCoordinate(msh)
        self.w = x[0] if self.dim == 1 else 1.0          # 1D: integrals carry r (per radian); 2D: per unit length
        pts = Q4.element.interpolation_points
        self.deps_expr = fem.Expression(self.eps4(self.Du), pts)
        self.deps = fem.Function(Q4)
        self.rq = fem.Function(Q1)                       # radius of each quadrature point
        self.rq.interpolate(fem.Expression(x[0] if self.dim == 1 else ufl.sqrt(x[0]**2 + x[1]**2), pts))
        v, du = ufl.TestFunction(self.V), ufl.TrialFunction(self.V)
        self.pload = fem.Constant(msh, 0.0)
        n = ufl.FacetNormal(msh)
        if self.dim == 2:
            Fext = -self.pload * ufl.dot(n, v) * self.ds(INNER)       # pressure: traction -p n on r = ri
        else:
            Fext = self.pload * self.ri * v * self.ds(INNER)
        Fint = ufl.dot(self.sig, self.eps4(v)) * self.w * self.dx
        self.F = fem.form(Fint - Fext if control == "pressure" else Fint)
        self.J = fem.form(ufl.dot(ufl.dot(self.Ct, self.eps4(du)), self.eps4(v)) * self.w * self.dx)
        # Dirichlet conditions: symmetry (2D); for displacement control also the inner surface
        fdim = self.dim - 1
        self.bcs = []
        if self.dim == 2:
            for tag, comp in ((BOTTOM, 1), (LEFT, 0)):
                dofs = fem.locate_dofs_topological(self.V.sub(comp), fdim, ft.find(tag))
                self.bcs.append(fem.dirichletbc(0.0, dofs, self.V.sub(comp)))
        self.g = fem.Function(self.V)                    # increment of the prescribed inner displacement
        if control == "displacement":
            dofs = fem.locate_dofs_topological(self.V, fdim, ft.find(INNER))
            self.bc_in = fem.dirichletbc(self.g, dofs)
            self.bcs.append(self.bc_in)
        # radial "lifting" field w_R = e_r (ro - r)/(ro - ri): the consistent reaction R(w_R) = p * ri * (pi/2 in 2D, 1 per radian in 1D)
        self.wR = fem.Function(self.V)
        if self.dim == 2:
            self.wR.interpolate(lambda X: np.vstack([X[0], X[1]]) / np.hypot(X[0], X[1]) * (ro - np.hypot(X[0], X[1])) / (ro - ri))
        else:
            self.wR.interpolate(lambda X: (ro - X[0]) / (ro - ri))
        self.R_form = fem.form(ufl.dot(self.sig, self.eps4(self.wR)) * self.w * self.dx)
        self.ur_form = fem.form((ufl.dot(self.u, -n) if self.dim == 2 else self.u) * self.ds(INNER))
        self.arc = (np.pi / 2 * ri) if self.dim == 2 else 1.0
        self.A = create_matrix(self.J); self.b = create_vector(self.V)
        self.ksp = PETSc.KSP().create(comm); self.ksp.setOperators(self.A)
        self.ksp.setType("preonly"); self.ksp.getPC().setType("lu"); self.ksp.getPC().setFactorSolverType("mumps")
        self.load, self.ref = 0.0, 0.0

    def eps4(self, u):
        """strain in Mandel form (xx, yy, zz, sqrt2 xy); 1D radial: (rr, theta theta, zz, 0). Plane strain: eps_zz = 0."""
        if self.dim == 2:
            e = ufl.sym(ufl.grad(u))
            return ufl.as_vector([e[0, 0], e[1, 1], 0, np.sqrt(2) * e[0, 1]])
        r = ufl.SpatialCoordinate(self.msh)[0]
        return ufl.as_vector([u.dx(0), u / r, 0, 0])

    def _constitutive(self):
        self.deps.interpolate(self.deps_expr)
        n4 = len(self.sig.x.array) // 4
        sig, p, Ct, pl = self.mat.update(self.deps.x.array.reshape(n4, 4), self.sig_n.x.array.reshape(n4, 4),
                                         self.p_n.x.array, self.tangent)
        self.sig.x.array[:] = sig.ravel(); self.p.x.array[:] = p; self.Ct.x.array[:] = Ct.ravel()
        return pl

    def _residual(self):
        """b = F(Du); constrained rows hold Du - (prescribed increment), so the first correction carries the boundary increment
        into the body with the converged tangent (lifting) instead of a jump in the boundary cells"""
        with self.b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(self.b, self.F)
        apply_lifting(self.b, [self.J], bcs=[self.bcs], x0=[self.Du.x.petsc_vec], alpha=-1.0)
        self.b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE)
        set_bc(self.b, self.bcs, self.Du.x.petsc_vec, -1.0)
        return self.b.norm()

    def step(self, load, rtol=1e-8, atol=1e-12, maxit=50):
        """one load step to `load` (pressure, or inner radial displacement). Newton on the increment Du from the last
        converged state; converged when |R| <= rtol * (largest first residual so far) or |R| <= atol.
        Returns (converged, iterations, residual history). Commits the state only if converged."""
        self.Du.x.array[:] = 0.0
        if self.control == "pressure":
            self.pload.value = load
        else:
            d = load - self.load                             # the increment of the inner radial displacement in this step
            if self.dim == 2:
                self.g.interpolate(lambda X: d * np.vstack([X[0], X[1]]) / np.hypot(X[0], X[1]))
            else:
                self.g.x.array[:] = d
        hist = []
        for it in range(maxit + 1):
            self._constitutive()
            res = self._residual(); hist.append(res)
            if it == 0:
                self.ref = max(self.ref, res)            # scale: the largest first residual of any step so far
            if res <= rtol * self.ref or res <= atol:
                break
            if it == maxit:
                return False, it, hist
            self.A.zeroEntries(); assemble_matrix(self.A, self.J, bcs=self.bcs); self.A.assemble()
            self.ksp.solve(self.b, self.du.x.petsc_vec); self.du.x.scatter_forward()
            if not np.all(np.isfinite(self.du.x.array)):
                return False, it, hist
            self.Du.x.array[:] -= self.du.x.array
        self.u.x.array[:] += self.Du.x.array                 # commit the converged step
        self.sig_n.x.array[:] = self.sig.x.array; self.p_n.x.array[:] = self.p.x.array
        self.load = load
        return True, it, hist

    def ur_inner(self):
        return comm.allreduce(fem.assemble_scalar(self.ur_form), op=MPI.SUM) / self.arc

    def pressure_from_reaction(self):
        """inner pressure from the consistent reaction (works for both controls; for pressure control it returns p)"""
        return comm.allreduce(fem.assemble_scalar(self.R_form), op=MPI.SUM) / (self.ri * self.arc)

    def plastic_radius(self):
        """largest radius of a quadrature point with p > 0 (ri if none is plastic)"""
        r = self.rq.x.array[self.p_n.x.array > 0.0]
        return comm.allreduce(r.max() if len(r) else self.ri, op=MPI.MAX)

    def max_von_mises(self):
        n4 = len(self.sig_n.x.array) // 4
        return comm.allreduce(von_mises(self.sig_n.x.array.reshape(n4, 4)).max(), op=MPI.MAX)

## Q1 — Theory (20 pts)

**(a) (7)** Backward Euler for J2 with linear isotropic hardening: $\boldsymbol\sigma=\boldsymbol\sigma^{\rm tr}-2\mu\,\Delta\boldsymbol\varepsilon^p$, $\Delta\boldsymbol\varepsilon^p=\Delta p\,\tfrac32\,\mathbf s/\sigma_{\rm eq}$
(the normal at the **end** of the step), $f(\boldsymbol\sigma,p_n+\Delta p)=0$. Show that $\mathbf s$ is parallel to $\mathbf s^{\rm tr}$, that $\sigma_{\rm eq}=\sigma_{\rm eq}^{\rm tr}-3\mu\Delta p$,
and derive $\Delta p$. What happens to $\operatorname{tr}\boldsymbol\sigma$?

**(b) (7)** Differentiate the update to obtain $\mathbb C^{\rm alg}$ (lecture §A4). Use $\partial\sigma^{\rm tr}_{\rm eq}/\partial\Delta\boldsymbol\varepsilon=2\mu\sqrt{3/2}\,\hat{\mathbf n}$ and
$\partial\hat{\mathbf n}/\partial\mathbf s^{\rm tr}=(\mathbb I-\hat{\mathbf n}\otimes\hat{\mathbf n})/|\mathbf s^{\rm tr}|$ (on deviators). Then compute $\mathbb C^{\rm alg}\hat{\mathbf n}$:
what stiffness does a plastic point have along the flow direction for $H=0$, and for $H>0$?

**(c) (6)** For the tube of the lab: why does the first-yield pressure depend on $\nu$ but the limit load not? Would $p_{\rm lim}$ change for a tube with closed
ends (generalized plane strain, Week 3)? Argue in two or three sentences, no calculation needed.

> *your answer*

## Q2 — Hardening (25 pts)

Linear isotropic hardening $H=E/10=7000$ MPa, otherwise the lab data.

**(a) (8)** Material point (an ordinary cell, `uniaxial` from `j2.py`): load in uniaxial stress to $\varepsilon_{xx}=5\varepsilon_y$, unload to **zero stress**, then
compress until the point yields again. Compare (i) the peak stress with the bilinear curve $\sigma_0+E_t(\varepsilon-\varepsilon_y)$, $E_t=EH/(E+H)$; (ii) the
remaining strain at zero stress with $p$; (iii) the stress at which reverse yielding starts with $-(\sigma_0+Hp)$. You must compute the strain at which the
stress is zero yourself (unloading is elastic).

In [ ]:
# TODO: Q2(a) — load, unload to sigma = 0, compress until reverse yielding; compare with the bilinear curve
import sys; sys.path.insert(0, "/tmp/fnx")
import numpy as np
from j2 import *

> *your comparison (Q2a)*

**(b) (10)** The tube with $H=E/10$: 2D ($8\times24$) and 1D (400 cells), 26 equal pressure steps to $1.3\,p_{\rm lim}$. Print the iterations per step,
$u_r(r_i)/u_y$ and the plastic radius at $p_{\rm lim}$ and at $1.3\,p_{\rm lim}$. At what pressure does the whole wall become plastic? Why is $p_{\rm lim}$ no longer a limit?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(b) — the hardening tube beyond p_lim, 2D and 1D

> *your results (Q2b)*

**(c) (7)** Newton's rate. For the 2D hardening tube, print the residual history (relative to the first residual) of the step that ends at $1.2\,p_{\rm lim}$, for the
consistent and the elastic tangent (`rtol=1e-10`, up to 400 iterations), and estimate the order $q\approx\log(r_{k+1}/r_k)/\log(r_k/r_{k-1})$ over the last iterations.
Compare the elastic-tangent rate with the perfectly plastic case of the lab.

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) — residual histories and the convergence order, both tangents

> *your answer (Q2c)*

## Q3 — Autofrettage (35 pts)

A thick tube is **autofrettaged**: overloaded once into the plastic range and unloaded. The residual stresses that remain make the bore compressive
and raise the pressure the tube can later carry elastically. Tube $r_i=1$, $r_o=3$ (plane strain), the lab's material with $H=0$. Use the 1D model
with 800 cells (fast and accurate), `-np 1`. Here $p_y=128.26$ MPa, $p_{\rm lim}=317.14$ MPa.

**(a) (12)** Overload to $p^*=0.75\,p_{\rm lim}$ (20 steps), unload to 0 (20 steps), reload to $p^*$ (20 steps). Show that (i) the unloading is elastic (no point's $p$
increases); (ii) the residual $u_r(r_i)$ equals $u_r(p^*)$ minus the Lamé displacement of $p^*$ — to how many digits, and why exactly?; (iii) the reloading
creates no new plastic strain. Print the residual hoop stress at the quadrature point nearest the bore, and at the outer surface. What has been gained?

In [ ]:
%%fenicsx -np 1
# TODO: Q3(a) — overload to 0.75 p_lim, unload, reload

> *your answers (Q3a)*

**(b) (13)** Larger overloads leave larger residual stresses — until the unloading itself yields the bore in **reverse**. Find, by bisection on $p^*/p_{\rm lim}$
to three digits, the largest overload for which the unloading stays elastic. Compare with the rule of thumb "reverse yielding if $p^*>2p_y$", which comes from
Tresca with $\nu=\tfrac12$. Explain in two sentences where the difference comes from.

In [ ]:
%%fenicsx -np 1
# TODO: Q3(b) — bisection for the onset of reverse yielding

> *your answer (Q3b)*

**(c) (10)** Overload beyond that limit, $p^*=0.95\,p_{\rm lim}$, and cycle $0\to p^*\to0$ three times. Print, for every half-cycle, the increase of $p$ at the bore and the residual
$u_r(r_i)$ after each unloading. Does the tube **shake down** (become elastic after the first cycle)? What does that mean for the design of the autofrettage pressure?

In [ ]:
%%fenicsx -np 1
# TODO: Q3(c) — three cycles at 0.95 p_lim

> *your answer (Q3c)*

## Q4 — Debugging an LLM-generated plasticity script (20 pts)

A student asked an LLM for "a self-contained DOLFINx 0.11 script for the thick tube with linear hardening, 1D radial model, loaded to $1.2\,p_{\rm lim}$ and
unloaded, with a return mapping in NumPy and its own Newton loop". The script below runs to the end without an error message and prints a number for every step. It has **four** defects (one of them needs a change in two lines).

**(a) (4)** Run it as it is and record the output.

**(b) (8)** List the four defects. For each one, name the **test** that exposes it (a material-point test, an elastic check against Lamé, a comparison with
`plas.py`, …) and show that test's output in (c).

**(c) (8)** Write the corrected script in the second cell and verify it: Lamé after two elastic steps, $f=0$ at the plastic points after the last loading step, and
agreement with `Tube` (1D, 400 cells, $H=E/10$) at $1.2\,p_{\rm lim}$ and after unloading.

In [ ]:
%%fenicsx -np 1
# LLM-generated (as received)
import numpy as np, ufl, basix.ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, create_matrix, create_vector, set_bc

comm = MPI.COMM_WORLD
E, nu, s0, H = 70e3, 0.3, 250.0, 7e3
ri, ro = 1.0, 1.3
mu, lmbda = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu))
one = np.array([1.0, 1.0, 1.0, 0.0]); C = lmbda * np.outer(one, one) + 2 * mu * np.eye(4)
p_lim = 2 * s0 / np.sqrt(3) * np.log(ro / ri)

msh = mesh.create_interval(comm, 400, [ri, ro])
V = fem.functionspace(msh, ("Lagrange", 2))
cell = msh.basix_cell()
Q4 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4,), degree=2))
Q44 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(4, 4), degree=2))
Q1 = fem.functionspace(msh, basix.ufl.quadrature_element(cell, value_shape=(), degree=2))
r = ufl.SpatialCoordinate(msh)[0]
eps4 = lambda w: ufl.as_vector([w.dx(0), w / r, 0, 0])
u, Du, du = fem.Function(V), fem.Function(V), fem.Function(V)
sig, sig_n, Ct, p, p_n, de = fem.Function(Q4), fem.Function(Q4), fem.Function(Q44), fem.Function(Q1), fem.Function(Q1), fem.Function(Q4)
de_expr = fem.Expression(eps4(u + Du), Q4.element.interpolation_points)

def update(deps, sn, pn):
    st = deps @ C
    s = st.copy(); m = (s[:, 0] + s[:, 1]) / 2; s[:, 0] -= m; s[:, 1] -= m
    seq = np.sqrt(1.5 * (s * s).sum(1))
    f = seq - (s0 + H * pn)
    dp = np.where(f > 0, f / (3 * mu), 0.0)
    nh = s / (np.sqrt(2 / 3) * np.where(seq > 0, seq, 1.0))[:, None]
    snew = st - (2 * mu * np.sqrt(1.5) * dp)[:, None] * nh
    b = 3 * mu * dp / np.where(seq > 0, seq, 1.0); tb = np.where(f > 0, 3 * mu / (3 * mu + H) - b, 0.0)
    Id = np.eye(4) - np.outer(one, one) / 3
    Cn = C - 2 * mu * (b[:, None, None] * Id + tb[:, None, None] * np.einsum("ni,nj->nij", nh, nh))
    return snew, pn + dp, Cn

v, w = ufl.TestFunction(V), ufl.TrialFunction(V)
fi = mesh.locate_entities_boundary(msh, 0, lambda x: np.isclose(x[0], ri))
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 0, fi, np.ones(len(fi), dtype=np.int32)))
pl = fem.Constant(msh, 0.0)
F = fem.form(ufl.dot(sig, eps4(v)) * r * ufl.dx - pl * ri * v * ds(1))
J = fem.form(ufl.dot(ufl.dot(Ct, eps4(w)), eps4(v)) * ufl.dx)
A = create_matrix(J); b = create_vector(V)
ksp = PETSc.KSP().create(comm); ksp.setOperators(A); ksp.setType("preonly"); ksp.getPC().setType("lu")
ur = fem.form(u * ds(1))

def newton(pv):
    pl.value = pv; Du.x.array[:] = 0.0
    for it in range(100):
        de.interpolate(de_expr); n4 = len(sig.x.array) // 4
        sn, pn, Cn = update(de.x.array.reshape(n4, 4), sig_n.x.array.reshape(n4, 4), p_n.x.array)
        sig.x.array[:] = sn.ravel(); p.x.array[:] = pn; Ct.x.array[:] = Cn.ravel()
        with b.localForm() as bl: bl.set(0.0)
        assemble_vector(b, F); b.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE)
        res = b.norm()
        if it == 0: r0 = res
        if res < 1e-8 * r0 or res < 1e-10: break
        A.zeroEntries(); assemble_matrix(A, J); A.assemble()
        ksp.solve(b, du.x.petsc_vec); du.x.scatter_forward(); Du.x.array[:] -= du.x.array
    u.x.array[:] += Du.x.array
    sig_n.x.array[:] = sig.x.array; p_n.x.array[:] = p.x.array
    return it

for pv in np.r_[np.linspace(0, 1.2 * p_lim, 13)[1:], np.linspace(1.2 * p_lim, 0, 7)[1:]]:
    it = newton(pv)
    print(f"p = {pv:7.2f} MPa  u_r(ri) = {fem.assemble_scalar(ur):.6e}  iterations {it}  max p = {p_n.x.array.max():.4e}")

In [ ]:
%%fenicsx -np 1
# TODO: Q4(c) — the corrected script and its verification

> *your list of defects and tests (Q4b), and your verification (Q4c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …